In [1]:
!git clone https://github.com/dea980/miniwatson_veichle.git


Cloning into 'miniwatson_veichle'...
remote: Enumerating objects: 1432, done.
remote: Counting objects: 100% (181/181), done.
remote: Compressing objects: 100% (136/136), done.
remote: Total 1432 (delta 64), reused 109 (delta 40), pack-reused 1251 (from 1)
Receiving objects: 100% (1432/1432), 73.48 MiB | 26.70 MiB/s, done.
Resolving deltas: 100% (628/628), done.


In [2]:
%cd miniwatson_veichle/ml/finetune

/content/miniwatson_veichle/ml/finetune


In [3]:
!pip install -U transformers peft trl bitsandbytes accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.6 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.13.0
    Uninstalling accelerate-1.13.0:
      Successfully uninstalled accelerate-1.13.0
  Attempting uninstall: transf

In [14]:
from google.colab import files
up = files.upload()   # 창 뜨면 로컬의 ml/data/out/train.jsonl, val.jsonl 둘 다 선택

Saving pref_seed.jsonl to pref_seed.jsonl


## 학습 로그 찍기
TRL/transformers가 빠르게 바뀌어서 또 다른 인자 에러(예: dataset_text_field, eval_strategy)가 날 수도 있어.

In [17]:
!sed -i 's/max_seq_length=args.max_seq/max_length=args.max_seq/' \
  /content/miniwatson_veichle/ml/finetune/train_qlora.py

In [18]:
!python /content/miniwatson_veichle/ml/finetune/train_qlora.py \
  --data /content/miniwatson_veichle/ml/finetune/train.jsonl --val /content/miniwatson_veichle/ml/finetune/val.jsonl \
  --base Qwen/Qwen2.5-7B-Instruct

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:   1% 2/339 [00:05<14:57,  2.66s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 339/339 [00:43<00:00,  7.75it/s]
trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/content/miniwatson_veichle/ml/finetune/train_qlora.py:76: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the 

In [24]:
!sed -i 's/ max_prompt_length=512, max_length=1024,//' \
  /content/miniwatson_veichle/ml/finetune/train_dpo.py

In [25]:
!python /content/miniwatson_veichle/ml/finetune/train_dpo.py \
  --data /content/miniwatson_veichle/ml/data/pref_seed.jsonl \
  --base Qwen/Qwen2.5-7B-Instruct

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:   1% 2/339 [00:06<18:19,  3.26s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 339/339 [00:47<00:00,  7.16it/s]
[dpo] SFT 어댑터 이어가기: adapters_7b
Adding EOS to train dataset: 100% 10/10 [00:00<00:00, 546.10 examples/s]
Tokenizing train dataset: 100% 10/10 [00:00<00:00, 377.31 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
{'train_runtime': '63.91', 'train_samples_per_second': '0.156', 'train_steps_per_second': '0.031', 'train_loss': '0.

# 서빙 (adapters_dpo → Ollama) — 단계별로

In [26]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
base = "Qwen/Qwen2.5-7B-Instruct"
m = AutoModelForCausalLM.from_pretrained(base, torch_dtype=torch.float16, device_map="auto")
m = PeftModel.from_pretrained(m, "adapters_dpo")   # ← SFT+DPO 합쳐진 최종 어댑터
m = m.merge_and_unload()
m.save_pretrained("vehicle-7b-dpo")
AutoTokenizer.from_pretrained(base).save_pretrained("vehicle-7b-dpo")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

가져오기 — HF Hub 권장 (merged fp16은 ~15GB라 tar 다운로드는 느려):

In [27]:
from huggingface_hub import login; login()   # HF 토큰
m.push_to_hub("YOUR_ID/vehicle-7b-dpo"); AutoTokenizer.from_pretrained(base).push_to_hub("YOUR_ID/vehicle-7b-dpo")

HfHubHTTPError: Client error '401 Unauthorized' for url 'https://huggingface.co/api/repos/create' (Request ID: Root=1-6a396e2c-602cb6c13a94c8eb012d3d75;abc1857c-f035-4d35-bf4a-4b330a453f8f)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

Invalid username or password.

로컬 M2: Q4 양자화 + Ollama 등록

In [28]:
huggingface-cli download YOUR_ID/vehicle-7b-dpo --local-dir ml/finetune/merged_dpo
printf 'FROM ./merged_dpo\n' > ml/finetune/Modelfile_dpo
ollama create vehicle-qwen2.5-7b-dpo --quantize q4_K_M -f ml/finetune/Modelfile_dpo
ollama list   # ~4.7GB 확인

SyntaxError: invalid decimal literal (124569543.py, line 1)